In [ ]:
"""
Open-Meteo Historical Weather Fetcher
--------------------------------------
Fetch hourly temperature and humidity for any location and date range.

Usage:
    python open_meteo_weather.py

Or import and call directly:
    from open_meteo_weather import get_historical_weather
"""

import urllib.request
import urllib.parse
import json
from datetime import datetime, date
import pandas as pd
import re

def get_historical_weather(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    temperature_unit: str = "celsius",
    timezone: str = "auto",
) -> dict:
    """
    Fetch hourly historical temperature and relative humidity from Open-Meteo.

    Parameters
    ----------
    latitude : float
        Latitude of the location (e.g. 40.7128 for New York City).
    longitude : float
        Longitude of the location (e.g. -74.0060 for New York City).
    start_date : str
        Start date in "YYYY-MM-DD" format (e.g. "2020-01-01").
        The API also accepts "YYYY-MM-DDTHH:MM" if you want to start at a
        specific hour, but plain dates are usually sufficient since the result
        already contains one row per hour.
    end_date : str
        End date in "YYYY-MM-DD" format (e.g. "2024-12-31").
    temperature_unit : str
        "celsius" (default) or "fahrenheit".
    timezone : str
        Timezone for the returned timestamps.  "auto" (default) infers it
        from the coordinates.  Pass any IANA tz string, e.g. "America/New_York".

    Returns
    -------
    dict with keys:
        "metadata"  – dict with location info and units
        "hourly"    – list of dicts, one per hour:
                        {
                          "datetime":    "2020-01-01T00:00",
                          "temperature": 3.2,          # °C or °F
                          "relative_humidity": 78      # %
                        }

    Raises
    ------
    ValueError  – bad parameter values
    RuntimeError – HTTP or API errors
    """

    # ── Validate dates ────────────────────────────────────────────────────────
    for label, value in (("start_date", start_date), ("end_date", end_date)):
        try:
            datetime.strptime(value[:10], "%Y-%m-%d")
        except ValueError:
            raise ValueError(f"{label} must be in YYYY-MM-DD format, got: {value!r}")

    if start_date > end_date:
        raise ValueError(
            f"start_date ({start_date}) must be before or equal to end_date ({end_date})."
        )

    if temperature_unit not in ("celsius", "fahrenheit"):
        raise ValueError("temperature_unit must be 'celsius' or 'fahrenheit'.")

    # ── Build URL ─────────────────────────────────────────────────────────────
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date[:10],   # keep only YYYY-MM-DD
        "end_date": end_date[:10],
        "hourly": "temperature_2m,relative_humidity_2m",
        "temperature_unit": temperature_unit,
        "timezone": timezone,
    }
    url = f"{base_url}?{urllib.parse.urlencode(params)}"

    # ── Make the request ──────────────────────────────────────────────────────
    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            raw = response.read().decode("utf-8")
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"HTTP {exc.code} from Open-Meteo: {exc.reason}") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Network error: {exc.reason}") from exc

    data = json.loads(raw)

    # Open-Meteo returns {"error": true, "reason": "..."} on bad requests
    if data.get("error"):
        raise RuntimeError(f"Open-Meteo API error: {data.get('reason')}")

    # ── Parse response ────────────────────────────────────────────────────────
    hourly = data["hourly"]
    times        = hourly["time"]
    temperatures = hourly["temperature_2m"]
    humidities   = hourly["relative_humidity_2m"]

    temp_label = "°C" if temperature_unit == "celsius" else "°F"

    records = []
    for dt, temp, hum in zip(times, temperatures, humidities):
        records.append(
            {
                "datetime": dt,
                "temperature": temp,          # may be None if data is missing
                "relative_humidity": hum,     # may be None if data is missing
            }
        )

    metadata = {
        "latitude":         data.get("latitude"),
        "longitude":        data.get("longitude"),
        "elevation_m":      data.get("elevation"),
        "timezone":         data.get("timezone"),
        "timezone_abbr":    data.get("timezone_abbreviation"),
        "temperature_unit": temp_label,
        "humidity_unit":    "%",
        "start_date":       start_date[:10],
        "end_date":         end_date[:10],
        "total_hours":      len(records),
    }

    return {"metadata": metadata, "hourly": records}


# ── Optional helpers ──────────────────────────────────────────────────────────

def filter_by_hour_range(records: list, start_hour: int, end_hour: int) -> list:
    """
    Keep only records whose hour falls within [start_hour, end_hour] (inclusive).

    Example – daytime only (8 AM–6 PM):
        daytime = filter_by_hour_range(result["hourly"], 8, 18)
    """
    filtered = []
    for r in records:
        hour = int(r["datetime"][11:13])
        if start_hour <= hour <= end_hour:
            filtered.append(r)
    return filtered


def to_csv(result: dict, filepath: str) -> None:
    """Write the hourly records to a CSV file."""
    import csv
    meta = result["metadata"]
    rows = result["hourly"]

    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        # Header comment rows
        # Column headers
        writer.writerow([
            "datetime",
            f"temperature ({meta['temperature_unit']})",
            f"relative_humidity ({meta['humidity_unit']})",
            f"# lat",  f"lon",
             f"elevation m",
             f"timezone",
             f'timezone_abbr',

        ])
        # Data
        for r in rows:
            writer.writerow([r["datetime"], r["temperature"], r["relative_humidity"],meta['latitude'], meta['longitude'],meta['elevation_m'],meta['timezone'], meta['timezone_abbr']])

    print(f"Saved {len(rows)} rows → {filepath}")


# ── Demo ──────────────────────────────────────────────────────────────────────


In [ ]:
def get_historical_weather_multiple(
    locations: list[tuple[str,str,float, float]],
    start_date: str,
    end_date: str,
    temperature_unit: str = "celsius",
    timezone: str = "auto",
    extra_col: str = "orig_team",
) -> dict[tuple, list[dict]]:
    """
    Fetch hourly temperature and relative humidity for one or more locations.

    Parameters
    ----------
    locations : list of (latitude, longitude) tuples
        E.g. [(40.71, -74.01), (41.88, -87.63)]
    start_date : str  "YYYY-MM-DD"
    end_date   : str  "YYYY-MM-DD"
    temperature_unit : "celsius" or "fahrenheit"
    timezone : IANA tz string e.g. "America/New_York", or "auto"

    Returns
    -------
    dict keyed by (latitude, longitude), each value a list of dicts:
        {"datetime": "2024-01-01T09:00", "temperature": 3.2, "relative_humidity": 72}
    """


    params = {
        "latitude":         ",".join(str(lat) for _,_,lat, _ in locations),
        "longitude":        ",".join(str(lon) for _,_,_, lon in locations),
        "start_date":       start_date,
        "end_date":         end_date,
        "hourly":           "temperature_2m,relative_humidity_2m,apparent_temperature",
        "temperature_unit": temperature_unit,
        "timezone":         timezone,
    }
    url = "https://archive-api.open-meteo.com/v1/archive?" + urllib.parse.urlencode(params)

    with urllib.request.urlopen(url, timeout=30) as r:
        data = json.loads(r.read().decode())

    if isinstance(data, dict):
        data = [data]
    for ind,dat in enumerate(data):
        dat['name'] = locations[ind][0]
        dat[extra_col] = locations[ind][1]
        
    return data



In [ ]:
dat

In [ ]:
team_camps =  pd.read_excel('../data_fifa.xlsx', sheet_name='teams-camps')
team_camps

In [ ]:
# data
# for location in data:
#         df = pd.DataFrame(location['hourly'])
#         df['time'] = pd.to_datetime(df['time'])

#         # Add scalar metadata as columns
#         scalar_keys = ['name','orig_team','latitude', 'longitude', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation']
#         for key in scalar_keys:
#             df[key] = location[key]
        
#         df.to_csv(out_fp + f'hourly_temp_{location['name']}.csv')

In [ ]:
# test_camp = camp_list[0:2]

# #data = get_historical_weather_multiple(test_camp, start_date, end_date)
# data[0]['hourly'].keys()

In [ ]:
stadiums =  pd.read_excel('../data_fifa.xlsx', sheet_name='stadiums')
stadiums

In [ ]:
def clean_filename(name):
    name = re.sub(r'[^\w\s-]', '', name)  # remove punctuation except hyphens
    name = name.strip().replace(' ', '_')  # replace spaces with underscores
    return name
def get_and_save_weather_data(start_date, end_date, loc_tups, out_fp, extra_col= "orig_teams"):
    
    data = get_historical_weather_multiple(loc_tups, start_date, end_date, extra_col=extra_col)
    for location in data:
        df = pd.DataFrame(location['hourly'])
        df['time'] = pd.to_datetime(df['time'])

        # Add scalar metadata as columns
        scalar_keys = ['name',extra_col,'latitude', 'longitude', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation']
        for key in scalar_keys:
            df[key] = location[key]
        df.to_csv(out_fp + f'hourly_temp_{clean_filename(location["name"])}.csv')
#Team data
start_date = "2021-06-01"
end_date = "2025-07-19"
camp_list = list(zip(team_camps['camp_name'],team_camps['team'],team_camps['camp_lat'], team_camps['camp_long']))
out_fp = 'datasets/camps/_hourly/'
#get_and_save_weather_data(start_date, end_date, camp_list, out_fp)

In [ ]:
stadiums= pd.read_csv('stadiums.csv')
stadiums

In [ ]:
start_date = "2021-06-01"
end_date = "2025-07-19"
stad_list = list(zip(stadiums['stadium'],stadiums['city_name'],stadiums['lat'], stadiums['lon']))
out_fp = 'datasets/stadiums/stadiums_hourly/'
#get_and_save_weather_data(start_date, end_date, stad_list, out_fp,extra_col="city")